# RAGAS: Evaluating RAG / Agentic Pipelines

Classic metrics (BLEU/ROUGE/METEOR) compare text to a fixed reference. **RAGAS** instead uses an LLM as a judge to score a RAG pipeline's behaviour along axes that don't require an exact reference string:

| Metric | Question it answers | Needs ground truth? |
|---|---|---|
| **Faithfulness** | Is the answer supported by the retrieved context (no hallucination)? | No |
| **Answer Relevancy** | Does the answer actually address the question asked? | No |
| **Context Precision** | Of the retrieved chunks, how many were actually relevant? | Uses ground truth to rank relevance |
| **Context Recall** | Did retrieval pull in everything needed to answer? | Yes |
| **Answer Correctness** | How factually/semantically close is the answer to the ground truth? | Yes |

This decomposition is what makes RAGAS useful in production: a low score tells you **which stage failed** (retrieval vs. generation), not just that "the answer was wrong".

> RAGAS scores things with an LLM judge, so it needs an LLM + embeddings configured (OpenAI by default, but any LangChain-compatible LLM works). Set `OPENAI_API_KEY` below, or swap in another provider in the "Configure the judge LLM" cell.

In [ ]:
# Install dependencies (uncomment if running fresh)
%pip install ragas datasets langchain-openai pandas matplotlib -q

In [ ]:
import os

# os.environ["OPENAI_API_KEY"] = "sk-..."  # set your key here, or export it in your shell before launching Jupyter
assert os.environ.get("OPENAI_API_KEY"), (
    "RAGAS needs a judge LLM. Set OPENAI_API_KEY (or reconfigure the LLM/embeddings "
    "cell below to point at another LangChain-compatible provider) before running the evaluate() cell."
)

## 1. Synthetic RAG dataset

We simulate the output of a RAG pipeline: for each `question` we fabricate the `contexts` a retriever returned (some relevant, some noisy/irrelevant), the `answer` a generator produced from those contexts (sometimes faithful, sometimes hallucinated or off-topic), and the `ground_truth` a human would write.

Cases covered:
- **good**: relevant context, faithful + correct answer
- **hallucination**: relevant context retrieved, but answer adds unsupported facts
- **retrieval_failure**: irrelevant context retrieved, so the answer is wrong/vague even though generation "tried its best"
- **partial**: relevant context, answer is correct but incomplete
- **noisy_context**: mix of relevant + irrelevant chunks retrieved, correct answer produced anyway

In [ ]:
from datasets import Dataset

synthetic_rows = [
    {
        "case": "good",
        "question": "What year was the Eiffel Tower completed?",
        "contexts": [
            "The Eiffel Tower was completed in 1889 as the entrance arch for the World's Fair held in Paris.",
            "It was designed by engineer Gustave Eiffel's company.",
        ],
        "answer": "The Eiffel Tower was completed in 1889.",
        "ground_truth": "The Eiffel Tower was completed in 1889.",
    },
    {
        "case": "hallucination",
        "question": "What year was the Eiffel Tower completed?",
        "contexts": [
            "The Eiffel Tower was completed in 1889 as the entrance arch for the World's Fair held in Paris.",
        ],
        "answer": "The Eiffel Tower was completed in 1887 and designed by Leonardo da Vinci.",
        "ground_truth": "The Eiffel Tower was completed in 1889.",
    },
    {
        "case": "retrieval_failure",
        "question": "What year was the Eiffel Tower completed?",
        "contexts": [
            "The Statue of Liberty was dedicated in 1886 and was a gift from France to the United States.",
            "It is located on Liberty Island in New York Harbor.",
        ],
        "answer": "Based on the available information, this appears to relate to a 19th century monument, though the exact completion year isn't specified.",
        "ground_truth": "The Eiffel Tower was completed in 1889.",
    },
    {
        "case": "partial",
        "question": "What causes seasons on Earth?",
        "contexts": [
            "Earth's axial tilt of about 23.5 degrees relative to its orbital plane causes seasons.",
            "As Earth orbits the Sun, different hemispheres receive more or less direct sunlight throughout the year.",
        ],
        "answer": "Seasons are caused by the tilt of the Earth's axis.",
        "ground_truth": "Seasons are caused by the Earth's axial tilt of about 23.5 degrees, which changes how directly sunlight hits each hemisphere as Earth orbits the Sun throughout the year.",
    },
    {
        "case": "noisy_context",
        "question": "What causes seasons on Earth?",
        "contexts": [
            "Earth's axial tilt of about 23.5 degrees relative to its orbital plane causes seasons.",
            "The Moon causes tides on Earth through gravitational pull.",
            "Mars has two small moons named Phobos and Deimos.",
        ],
        "answer": "Seasons occur because Earth's axis is tilted about 23.5 degrees, changing how directly sunlight reaches each hemisphere over the year.",
        "ground_truth": "Seasons are caused by the Earth's axial tilt of about 23.5 degrees, which changes how directly sunlight hits each hemisphere as Earth orbits the Sun throughout the year.",
    },
]

ragas_dataset = Dataset.from_list(synthetic_rows)
ragas_dataset.to_pandas()

## 2. Configure the judge LLM

RAGAS wraps a LangChain chat model + embeddings model to do the actual scoring. Swap `ChatOpenAI`/`OpenAIEmbeddings` below for any other LangChain-compatible provider (Azure OpenAI, Anthropic via `langchain-anthropic`, local models via Ollama, etc.) if you're not using OpenAI.

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

judge_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))
judge_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

## 3. Run RAGAS evaluation

- `faithfulness` and `answer_relevancy` only need `question`, `answer`, `contexts`.
- `context_precision`, `context_recall`, `answer_correctness` additionally need `ground_truth`.

In [ ]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
    answer_correctness,
)

metrics = [faithfulness, answer_relevancy, context_precision, context_recall, answer_correctness]

result = evaluate(
    ragas_dataset,
    metrics=metrics,
    llm=judge_llm,
    embeddings=judge_embeddings,
)

result_df = result.to_pandas()
result_df

## 4. Inspect per-case scores

In [ ]:
import pandas as pd

pd.set_option("display.max_colwidth", 80)
score_cols = ["faithfulness", "answer_relevancy", "context_precision", "context_recall", "answer_correctness"]
display_df = result_df[["case"] + score_cols].round(3) if "case" in result_df.columns else result_df[score_cols].round(3)
display_df

In [ ]:
import matplotlib.pyplot as plt

ax = display_df.set_index("case")[score_cols].plot(kind="bar", figsize=(12, 5))
ax.set_title("RAGAS metrics across synthetic RAG cases")
ax.set_ylabel("score")
ax.set_ylim(0, 1)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## Expected pattern (what to look for)

- **`good`**: high across the board.
- **`hallucination`**: low `faithfulness` (answer isn't supported by context) despite context being relevant — tells you it's a **generation** problem, not retrieval.
- **`retrieval_failure`**: low `context_precision`/`context_recall` — tells you it's a **retrieval** problem; the generator can't be blamed for a vague answer given useless context.
- **`partial`**: high faithfulness/relevancy but lower `answer_correctness` (missing detail vs. ground truth).
- **`noisy_context`**: `context_precision` dips (irrelevant chunks retrieved) even though the final answer is fine — the pipeline got lucky, and this is exactly the kind of case that regresses under a different question phrasing.

This per-stage breakdown — rather than a single pass/fail score — is why RAGAS (and LLM-as-judge rubric scoring generally) is the standard approach corporate teams use for RAG/agentic pipelines, layered on top of golden-set regression tests in CI.